In [1]:
import pandas as pd

df = pd.read_csv("../PLN/data/noticias_games (2).csv")
df.head()

,url,titulo,descricao,autor,tags,texto,data
0,https://www.eurogamer.pt/tecmo-bowl-sera-adapt...,Tecmo Bowl será adaptado para filme,"Tecmo Bowl, clássico da Koei Tecmo lançado no ...",NaN,"Android, iOS","Tecmo Bowl, clássico da Koei Tecmo lançado no ...",2026-02-03T16:35:31.309576+00:00
1,https://www.eurogamer.pt/fas-adoram-super-mari...,Fãs adoram Super Mario Galaxy: O Filme,Super Mario Galaxy: O Filme chegou finalmente ...,NaN,"Action Adventure, Platformer, Nintendo Switch,...",Super Mario Galaxy: O Filme chegou finalmente ...,2026-04-02T07:40:11.552201+00:00
2,https://www.eurogamer.pt/pokemon-firered-leafg...,Pokémon FireRed & LeafGreen inclui conteúdo au...,Pokémon FireRed & LeafGreen para a Nintendo Sw...,NaN,"Action Adventure, Pokémon FireRed and LeafGree...",Pokémon FireRed & LeafGreen para a Nintendo Sw...,2026-03-01T09:15:15.110566+00:00
3,https://www.eurogamer.pt/switch-2-ja-alcancou-...,Switch 2 já alcançou 58% das vendas da PS5 em ...,"O recente relatório do governo japonês, sobre ...",NaN,"Third person, Racing, Nintendo, Nintendo Switc...","O recente relatório do governo japonês, sobre ...",2026-02-09T13:33:14.368225+00:00
4,https://www.eurogamer.pt/produtora-de-slay-the...,Produtora de Slay the Spire 2 detesta micro-tr...,"Slay the Spire 2 chegou com grande sucesso, em...",NaN,"Multiplayer Cooperative, Indie, PC, Slay the S...","Slay the Spire 2 chegou com grande sucesso, em...",2026-03-10T14:42:15.059919+00:00


In [2]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["texto"].apply(limpar_texto)

df[["texto", "texto_limpo"]].head()

,texto,texto_limpo
0,"Tecmo Bowl, clássico da Koei Tecmo lançado no ...",tecmo bowl classico da koei tecmo lancado no f...
1,Super Mario Galaxy: O Filme chegou finalmente ...,super mario galaxy o filme chegou finalmente a...
2,Pokémon FireRed & LeafGreen para a Nintendo Sw...,pokemon firered leafgreen para a nintendo swit...
3,"O recente relatório do governo japonês, sobre ...",o recente relatorio do governo japones sobre o...
4,"Slay the Spire 2 chegou com grande sucesso, em...",slay the spire 2 chegou com grande sucesso em ...


In [3]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,tecmo bowl classico da koei tecmo lancado no f...,"[tecmo, bowl, classico, koei, tecmo, lancado, ...",tecmo bowl classico koei tecmo lancado final a...
1,super mario galaxy o filme chegou finalmente a...,"[super, mario, galaxy, filme, chegou, finalmen...",super mario galaxy filme chegou finalmente cin...
2,pokemon firered leafgreen para a nintendo swit...,"[pokemon, firered, leafgreen, nintendo, switch...",pokemon firered leafgreen nintendo switch incl...
3,o recente relatorio do governo japones sobre o...,"[recente, relatorio, governo, japones, sobre, ...",recente relatorio governo japones sobre gastos...
4,slay the spire 2 chegou com grande sucesso em ...,"[slay, the, spire, 2, chegou, grande, sucesso,...",slay the spire 2 chegou grande sucesso acesso ...


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,00,000,000mo,003,005,007,00h,01,011,012,...,zstd,zu,zuo,zur,zuwoyou,zwei,zwigqfdrl8,zygarde,zynga,zysppc5w4p
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:

colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]
df_bow = df_bow.drop(columns=colunas_com_numeros)

freq = df_bow.sum(axis=0)

colunas_validas = freq[freq > 2].index

df_bow = df_bow[colunas_validas]

print(f"{len(colunas_com_numeros)} colunas com números removidas")
print(f"{len(freq) - len(colunas_validas)} colunas raras removidas")

df_bow.head()

0 colunas com números removidas
12226 colunas raras removidas


,aaa,aang,aaron,abaixo,abandona,abandonado,abandonar,abandono,abandonou,abby,...,zona,zonas,zone,zootopia,zootropolis,zor,zoro,zur,zwei,zygarde
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
metadados = df[["data", "tags", "titulo", "url"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data,tags,titulo,url,bow_aa,bow_aaa,bow_aacs,bow_aang,bow_aaron,bow_ab,...,bow_zoroark,bow_zreik,bow_zstd,bow_zu,bow_zuo,bow_zur,bow_zuwoyou,bow_zwei,bow_zygarde,bow_zynga
0,2026-02-03T16:35:31.309576+00:00,"Android, iOS",Tecmo Bowl será adaptado para filme,https://www.eurogamer.pt/tecmo-bowl-sera-adapt...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2026-04-02T07:40:11.552201+00:00,"Action Adventure, Platformer, Nintendo Switch,...",Fãs adoram Super Mario Galaxy: O Filme,https://www.eurogamer.pt/fas-adoram-super-mari...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2026-03-01T09:15:15.110566+00:00,"Action Adventure, Pokémon FireRed and LeafGree...",Pokémon FireRed & LeafGreen inclui conteúdo au...,https://www.eurogamer.pt/pokemon-firered-leafg...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2026-02-09T13:33:14.368225+00:00,"Third person, Racing, Nintendo, Nintendo Switc...",Switch 2 já alcançou 58% das vendas da PS5 em ...,https://www.eurogamer.pt/switch-2-ja-alcancou-...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2026-03-10T14:42:15.059919+00:00,"Multiplayer Cooperative, Indie, PC, Slay the S...",Produtora de Slay the Spire 2 detesta micro-tr...,https://www.eurogamer.pt/produtora-de-slay-the...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 21406


bow_jogo         2049
bow_the          1129
bow_jogadores    1022
bow_jogos        1020
bow_ver           862
bow_serie         815
bow_nintendo      803
bow_novo          747
bow_vai           722
bow_of            721
dtype: int64

In [9]:
frequencia_palavras.tail(10)

bow_manuais        1
bow_manto          1
bow_zu             1
bow_zstd           1
bow_zreik          1
bow_zords          1
bow_zofia          1
bow_abas           1
bow_abandons       1
bow_abandonarem    1
dtype: int64

In [10]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

jogo         738
ver          610
ainda        500
jogos        498
novo         493
jogadores    483
vai          471
the          455
youtube      411
nova         408
dtype: int64

In [11]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
1132,Nintendo Direct: Partner Showcase - vê em direto,30
11,A Marvel começa a lembrar-se que os X-Men existem,34
833,Nacon revela novidades em março,35
653,Primeiras imagens de Pokémon Winds & Waves,37
970,Lords of the Fallen vendeu 2.5 milhões de cópias,38
984,Nintendo Treehouse: Live anunciada,40
748,Pokémon XD: Gale of Darkness chegará à Switch 2,41
325,Digimon Story Time Stranger a 60fps na Switch 2,41
931,JRPGs da Square Enix continuam a conquistar jo...,41
992,Nova Super Mario Galaxy: O Filme Direct anunciada,41
